In [ ]:
!pip install torch torchvision --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Kullanılan cihaz: {device}")

Kullanılan cihaz: cuda


In [ ]:
class BasitCNN(nn.Module):
    """
    Basit bir görüntü sınıflandırma CNN'i.
    Giriş: (batch, 1, 28, 28) boyutunda gri tonlamalı görüntü
    Çıkış: (batch, 10) boyutunda sınıf skorları (logits)
    """
    def __init__(self, num_classes=10):
        super().__init__()

        # --- 1. Konvolüsyon Bloğu ---
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)

        # --- 2. Konvolüsyon Bloğu ---
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # İki kez pool uygulanacağı için: 28 -> 14 -> 7
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.pool(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.pool(x)

        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)

        return x

In [ ]:
# Modeli oluşturup cihaza (CPU/GPU) taşıyoruz
model_cnn = BasitCNN(num_classes=10).to(device)
print(model_cnn)

# Sahte (dummy) bir görüntü batch'i ile ileri yayılımı (forward pass) test edelim
sahte_girdi = torch.randn(8, 1, 28, 28).to(device)
cikti = model_cnn(sahte_girdi)

print(f"Girdi boyutu : {sahte_girdi.shape}")
print(f"Çıktı boyutu : {cikti.shape}  # (batch_size=8, num_classes=10)")

BasitCNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=1568, out_features=128, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)
Girdi boyutu : torch.Size([8, 1, 28, 28])
Çıktı boyutu : torch.Size([8, 10])  # (batch_size=8, num_classes=10)


In [ ]:
# Kayıp fonksiyonu: çok sınıflı sınıflandırma için standart seçim
criterion = nn.CrossEntropyLoss()

# Optimizer: ağırlıkları gradyanlara göre güncelleyen algoritma
optimizer = torch.optim.Adam(model_cnn.parameters(), lr=1e-3)

# --- Tek bir eğitim adımı örneği (sahte veriyle) ---
sahte_etiketler = torch.randint(0, 10, (8,)).to(device)  # 8 örnek için rastgele doğru sınıflar

optimizer.zero_grad()                    # Önceki adımdan kalan gradyanları sıfırla
cikti = model_cnn(sahte_girdi)           # İleri yayılım: tahminleri hesapla
loss = criterion(cikti, sahte_etiketler) # Tahmin ile gerçek etiket arasındaki hatayı hesapla
loss.backward()                          # Geri yayılım: her ağırlık için gradyanı hesapla
optimizer.step()                         # Ağırlıkları gradyana göre güncelle

print(f"Örnek eğitim adımı kaybı (loss): {loss.item():.4f}")

Örnek eğitim adımı kaybı (loss): 2.5416


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Görüntüleri tensöre çevirip normalize ediyoruz
# MNIST'in bilinen ortalama (mean) ve standart sapma (std) değerleri
donusum = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

egitim_verisi = datasets.MNIST(root="./data", train=True, download=True, transform=donusum)
test_verisi = datasets.MNIST(root="./data", train=False, download=True, transform=donusum)

egitim_loader = DataLoader(egitim_verisi, batch_size=64, shuffle=True)
test_loader = DataLoader(test_verisi, batch_size=64, shuffle=False)

print(f"Eğitim örneği sayısı: {len(egitim_verisi)}")
print(f"Test örneği sayısı  : {len(test_verisi)}")

100%|██████████| 9.91M/9.91M [00:00<00:00, 19.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 486kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.49MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 15.3MB/s]

Eğitim örneği sayısı: 60000
Test örneği sayısı  : 10000


In [ ]:
# Önceki model bir kez "sahte" veriyle güncellenmişti, temiz başlıyoruz
model_cnn = BasitCNN(num_classes=10).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_cnn.parameters(), lr=1e-3)

In [ ]:
EPOCH_SAYISI = 3  # Tüm veri setinin kaç kez baştan sona görüleceği

for epoch in range(EPOCH_SAYISI):
    model_cnn.train()  # Modeli eğitim moduna al (dropout/batchnorm aktif)
    toplam_loss = 0.0

    for gorseller, etiketler in egitim_loader:
        gorseller, etiketler = gorseller.to(device), etiketler.to(device)

        optimizer.zero_grad()
        cikti = model_cnn(gorseller)
        loss = criterion(cikti, etiketler)
        loss.backward()
        optimizer.step()

        toplam_loss += loss.item()

    ortalama_loss = toplam_loss / len(egitim_loader)
    print(f"Epoch {epoch+1}/{EPOCH_SAYISI} - Ortalama Loss: {ortalama_loss:.4f}")

Epoch 1/3 - Ortalama Loss: 0.1724
Epoch 2/3 - Ortalama Loss: 0.0644
Epoch 3/3 - Ortalama Loss: 0.0507


In [ ]:
model_cnn.eval()  # Değerlendirme moduna al (dropout kapanır, batchnorm sabitlenir)

dogru_sayisi = 0
toplam_sayisi = 0

with torch.no_grad():  # Test sırasında gradyan hesaplamaya gerek yok
    for gorseller, etiketler in test_loader:
        gorseller, etiketler = gorseller.to(device), etiketler.to(device)

        cikti = model_cnn(gorseller)
        tahminler = torch.argmax(cikti, dim=1)

        dogru_sayisi += (tahminler == etiketler).sum().item()
        toplam_sayisi += etiketler.size(0)

dogruluk = 100 * dogru_sayisi / toplam_sayisi
print(f"Test doğruluğu: {dogruluk:.2f}%  ({dogru_sayisi}/{toplam_sayisi})")

Test doğruluğu: 98.81%  (9881/10000)
